# Notebook 05: Autocorrelation and Error Analysis

**Learning objectives:**
- Understand why consecutive Monte Carlo measurements are correlated
- Compute the autocorrelation function and integrated autocorrelation time
- Learn why naive error estimates underestimate uncertainty
- Apply binning analysis as an alternative to autocorrelation
- Determine optimal measurement separation

**Prerequisites:** Notebook 04 (Monte Carlo thermalization)



In [ ]:
from notebook_utils import setup_paths, quick_metropolis, autocorrelation
setup_paths()

import numpy as np
import matplotlib.pyplot as plt
import su2

## 1. Why Are Monte Carlo Measurements Correlated?

Each Metropolis sweep only makes small changes to the gauge field.
Consecutive configurations are therefore similar, and measurements
taken on them are **correlated**. Only after many sweeps does the
configuration "forget" its past — this decorrelation time $\tau$ sets
the number of truly independent measurements.

If we ignore correlations and naively compute the error as
$\sigma_{\rm naive} = \sigma / \sqrt{N}$, we **underestimate** the true
uncertainty. The corrected error is:
$$\sigma_{\rm correct} = \sigma\,\sqrt{\frac{2\,\tau_{\rm int}}{N}}
= \sigma_{\rm naive} \times \sqrt{2\,\tau_{\rm int}}$$

This is why we need autocorrelation analysis: to know how much to
**inflate our error bars** beyond the naive estimate.

In [ ]:
# Generate a long plaquette time series
La = [4, 4, 4, 4]
beta = 2.4
n_sweeps = 500

plaqs, _ = quick_metropolis(La, beta=beta, n_sweeps=n_sweeps, start='cold', seed=42)

# Discard thermalization (first 50 sweeps)
plaqs_eq = plaqs[50:]

plt.figure(figsize=(10, 3))
plt.plot(plaqs_eq, linewidth=0.8)
plt.xlabel('Sweep (after thermalization)')
plt.ylabel(r'$\langle P \rangle$')
plt.title(f'Plaquette time series ($\\beta = {beta}$, $4^4$ lattice)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Mean plaquette: {np.mean(plaqs_eq):.6f}")
print(f"Std deviation:  {np.std(plaqs_eq):.6f}")

## 2. The Autocorrelation Function

The normalized autocorrelation function measures how correlated
measurements separated by $\Delta$ sweeps are:

$$\rho(\Delta) = \frac{\langle P(n) P(n+\Delta) \rangle - \langle P \rangle^2}
{\langle P^2 \rangle - \langle P \rangle^2}$$

By definition $\rho(0) = 1$ and $\rho(\Delta) \to 0$ for $\Delta \gg \tau$.

In [ ]:
# Compute autocorrelation
max_lag = 80
rho = autocorrelation(plaqs_eq, max_lag=max_lag)

plt.figure(figsize=(8, 4))
plt.plot(range(max_lag), rho, 'o-', markersize=3)
plt.axhline(0, color='k', linewidth=0.5)
plt.axhline(0.1, color='r', ls='--', alpha=0.5, label=r'$\rho = 0.1$')
plt.xlabel(r'Lag $\Delta$ (sweeps)')
plt.ylabel(r'$\rho(\Delta)$')
plt.title('Autocorrelation function of plaquette')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Find where rho drops below 0.1
below_01 = np.where(rho < 0.1)[0]
if len(below_01) > 0:
    print(f"Autocorrelation drops below 0.1 at lag = {below_01[0]} sweeps")

## 3. Integrated Autocorrelation Time

The **integrated autocorrelation time** summarizes the correlation
into a single number:

$$\tau_{\rm int} = \frac{1}{2} + \sum_{\Delta=1}^{W} \rho(\Delta)$$

where $W$ is a cutoff (typically where $\rho$ first becomes consistent
with zero). The effective number of independent measurements is:

$$N_{\rm eff} = \frac{N}{2\,\tau_{\rm int}}$$

In [ ]:
# Compute integrated autocorrelation time with automatic windowing
def compute_tau_int(rho, c=5.0):
    """Compute tau_int with automatic window (Madras & Sokal criterion).
    
    The window W is chosen as the smallest value where W >= c * tau_int(W).
    """
    tau = 0.5
    for W in range(1, len(rho)):
        tau += rho[W]
        if W >= c * tau:
            return tau, W
    return tau, len(rho) - 1

tau_int, W = compute_tau_int(rho)
N = len(plaqs_eq)
N_eff = N / (2 * tau_int)

print(f"Integrated autocorrelation time: tau_int = {tau_int:.2f} sweeps")
print(f"Window used: W = {W}")
print(f"Total measurements: N = {N}")
print(f"Effective independent measurements: N_eff = {N_eff:.1f}")
print(f"Naive error would be off by factor sqrt(2*tau_int) = {np.sqrt(2*tau_int):.2f}")

## 4. Naive vs Correct Errors

The naive standard error $\sigma_{\rm naive} = \sigma / \sqrt{N}$ ignores
correlations. The correct error accounts for them:

$$\sigma_{\rm correct} = \sigma \sqrt{\frac{2\tau_{\rm int}}{N}}
= \sigma_{\rm naive} \times \sqrt{2\tau_{\rm int}}$$

In [ ]:
sigma = np.std(plaqs_eq)
err_naive = sigma / np.sqrt(N)
err_correct = sigma * np.sqrt(2 * tau_int / N)

print(f"Standard deviation:  sigma = {sigma:.6f}")
print(f"Naive error:         {err_naive:.6f}")
print(f"Corrected error:     {err_correct:.6f}")
print(f"Ratio (correction):  {err_correct / err_naive:.2f}x")
print(f"\nPlaquette = {np.mean(plaqs_eq):.6f} +/- {err_correct:.6f} (corrected)")

## 5. Binning Analysis

An alternative to computing $\tau_{\rm int}$ directly is **binning**:
group consecutive measurements into bins of size $B$, average within
each bin, and compute the error of the bin averages.

**Why does this work?** By averaging consecutive measurements into bins,
correlations *within* each bin are absorbed into the bin average. Once the
bin size exceeds $2\tau_{\rm int}$, the bin averages become approximately
independent, and the naive error formula $\sigma / \sqrt{N_{\rm bins}}$
gives the correct result — no autocorrelation correction needed.

As $B$ increases past $2\tau_{\rm int}$, the error estimate plateaus at
the correct value. This plateau should agree with the $\tau_{\rm int}$
method from Section 3.

In [ ]:
bin_sizes = [1, 2, 5, 10, 15, 20, 30, 50, 75, 100]
bin_errors = []

for B in bin_sizes:
    n_bins = len(plaqs_eq) // B
    if n_bins < 2:
        bin_errors.append(np.nan)
        continue
    bins = plaqs_eq[:n_bins * B].reshape(n_bins, B).mean(axis=1)
    bin_errors.append(np.std(bins) / np.sqrt(n_bins))

plt.figure(figsize=(8, 4))
plt.plot(bin_sizes, bin_errors, 'o-', markersize=6)
plt.axhline(err_naive, color='b', ls=':', label=f'Naive error = {err_naive:.5f}')
plt.axhline(err_correct, color='r', ls='--', label=f'Corrected = {err_correct:.5f}')
plt.xlabel('Bin size $B$ (sweeps)')
plt.ylabel('Error estimate')
plt.title('Binning analysis: error vs bin size')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("The error plateaus once the bin size exceeds 2*tau_int")
print(f"2*tau_int ~ {2*tau_int:.0f} sweeps")

## 6. Practical Recipe

**How many sweeps between measurements?**

A rule of thumb: separate measurements by at least $2\tau_{\rm int}$ sweeps.
This ensures approximately independent samples.

For production runs:
1. Run a short pilot simulation (e.g. 500 sweeps)
2. Compute $\tau_{\rm int}$ from the plaquette
3. Set measurement interval to $\geq 2\tau_{\rm int}$
4. Adjust if other observables (like the topological charge) have longer autocorrelation times

## 7. Runtime Scaling

Monte Carlo simulations scale steeply with lattice volume. Each sweep
must visit all $V = L^4$ sites, and each site requires computing staples
(summing over neighbors). Doubling $L$ increases the volume by $2^4 = 16\times$.

To minimize runtime while keeping independent measurements, **skip sweeps**:
measure every $2\tau_{\rm int}$ sweeps instead of every sweep. This way each
measurement is approximately independent, and you avoid wasting time on
correlated data.

In [ ]:
import time

# Compare runtime: 4^4 vs 8^4 lattice (small number of sweeps)
n_test = 10

print("Runtime comparison (10 sweeps each):\n")

t0 = time.time()
quick_metropolis([4, 4, 4, 4], beta=2.4, n_sweeps=n_test, start='cold', seed=1)
t_4 = time.time() - t0
print(f"\n  4^4 lattice (V = {4**4}):  {t_4:.1f} seconds")

t0 = time.time()
quick_metropolis([8, 8, 8, 8], beta=2.4, n_sweeps=n_test, start='cold', seed=1)
t_8 = time.time() - t0
print(f"\n  8^4 lattice (V = {8**4}): {t_8:.1f} seconds")

print(f"\n  Ratio: {t_8/t_4:.1f}x  (volume ratio = {8**4/4**4:.0f}x)")

In [ ]:
# Compare autocorrelation at different beta values
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, b in zip(axes, [1.5, 2.4, 4.0]):
    plaqs_b, _ = quick_metropolis(La, beta=b, n_sweeps=300,
                                  start='cold', seed=42)
    plaqs_b_eq = plaqs_b[50:]
    rho_b = autocorrelation(plaqs_b_eq, max_lag=60)
    tau_b, _ = compute_tau_int(rho_b)
    
    ax.plot(range(len(rho_b)), rho_b, 'o-', markersize=3)
    ax.axhline(0, color='k', linewidth=0.5)
    ax.set_xlabel(r'Lag $\Delta$')
    ax.set_ylabel(r'$\rho(\Delta)$')
    ax.set_title(f'$\\beta = {b}$, $\\tau_{{\\rm int}} = {tau_b:.1f}$')
    ax.grid(True, alpha=0.3)

plt.suptitle('Autocorrelation at different coupling strengths', fontsize=13)
plt.tight_layout()
plt.show()

## Exercises

1. **$\tau_{\rm int}$ at different $\beta$**: Compute $\tau_{\rm int}$ at
   $\beta = 1.0, 1.5, 2.0, 2.4, 3.0, 4.0$. Plot $\tau_{\rm int}$ vs $\beta$.
   Does the autocorrelation time increase or decrease with $\beta$?
   (*Near a phase transition, $\tau_{\rm int}$ diverges — this is called
   critical slowing down.*)

2. **Binning verification**: Using the binning analysis from Section 5,
   verify that the error plateaus at the same value predicted by
   $\tau_{\rm int}$: $\sigma_{\rm correct} = \sigma_{\rm naive}\sqrt{2\tau_{\rm int}}$.
   Do both methods agree? They should — if not, check your window
   cutoff or bin sizes.

3. **Optimal separation**: From your $\tau_{\rm int}$ measurement at $\beta = 2.4$,
   determine the minimum number of sweeps between independent measurements.
   If you wanted 100 independent configurations, how many total sweeps
   would you need (including thermalization)?